In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt

from calc_flatfield import *
import glob
from kll import kll

In [2]:
files = sorted(glob.glob('out/*'))
files

['out/phi-fdt-ilam_20260310T040003_V202608171053C_0663100100.fits',
 'out/phi-fdt-ilam_20260310T040405_V202608171054C_0663100125.fits',
 'out/phi-fdt-ilam_20260310T040805_V202608171055C_0663100150.fits',
 'out/phi-fdt-ilam_20260310T041205_V202608171055C_0663100175.fits',
 'out/phi-fdt-ilam_20260310T041605_V202608171056C_0663100200.fits',
 'out/phi-fdt-ilam_20260310T042005_V202608171057C_0663100225.fits',
 'out/phi-fdt-ilam_20260310T042405_V202608171057C_0663100250.fits',
 'out/phi-fdt-ilam_20260310T042805_V202608171058C_0663100275.fits',
 'out/phi-fdt-ilam_20260310T043205_V202608171059C_0663100300.fits']

In [3]:
shifts = []
images = []
centers = []

for file in files:
    with fits.open(file) as hdul:
        header = hdul[0].header
        data = hdul[0].data

    contpos = int(header['CONTPOS']) - 1
    xc = header['CRPIX2'] - 1
    yc = header['CRPIX1'] - 1

    images += [data[contpos,0].copy()]
    centers += [(xc, yc)]

centers = np.array(centers)
images = np.array(images)

In [4]:
plt.figure(figsize=(10,10))
plt.imshow(images[4], cmap='gray')
plt.tight_layout()

In [19]:
transmittance = kll(images, np.round(centers).astype(int), images.clip(0), niter=100, sigma=100, slope=True, vmin=0.1, vmax=2)

In [25]:
plt.figure(figsize=(10,10))
plt.imshow(transmittance, cmap='gray', vmin=0.8, vmax=1.1)
plt.tight_layout()

In [16]:
transmittance_ = kll(images, np.round(centers).astype(int), images.clip(0), niter=100, sigma=None, slope=True, vmin=0.1, vmax=2)

In [17]:
plt.figure(figsize=(10,10))
plt.imshow(transmittance_, cmap='gray', vmin=0.8, vmax=1.1)
plt.tight_layout()

In [26]:
temp = transmittance_ - transmittance
temp -= np.nanmedian(temp[900:1100,900:1100])

plt.figure(figsize=(10,10))
plt.imshow(temp, cmap='gray', vmin=-0.01, vmax=0.01)
plt.tight_layout()

In [29]:
plt.figure(figsize=(10,10))
plt.imshow(images[1] / transmittance, cmap='gray')
plt.tight_layout()